# Sesión 3 — Entrenar, probar y no hacerse trampa

**Introducción al Análisis Avanzado de Datos con Python** · Univalle · Módulo 1 (cierre)

## Objetivos de hoy
Al terminar la sesión usted podrá:
1. Explicar para qué se separan entrenamiento y prueba, y hacerlo bien (`train_test_split` con `stratify`).
2. Diagnosticar **sobreajuste** comparando el accuracy de entrenamiento con el de prueba, y elegir la complejidad del modelo.
3. Detectar una **fuga de información** (*data leakage*) y calcular cualquier estadística **solo con entrenamiento**.
4. Seguir una predicción individual por dentro del árbol y explicar qué optimiza su `.fit()`.

> 💾 **Antes de escribir nada:** menú *Archivo → Guardar una copia en Drive* y trabaje en esa copia; si no, lo que escriba se pierde al cerrar la pestaña. Si Colab avisa que el notebook no es de Google, elija *Ejecutar de todos modos*.

## Dónde estamos en el pipeline

```
 ┌────────┐   ┌──────────────────┐   ┌────────┐   ┌────────────┐   ┌────────────────┐
 │ DATOS  │ → │ PREPROCESAMIENTO │ → │ MODELO │ → │ EVALUACIÓN │ → │ INTERPRETACIÓN │
 └────────┘   └──────────────────┘   └────────┘   └────────────┘   └────────────────┘
                                       ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲
   HOY: entrenar y medir de forma honesta. Cierra el Módulo 1 y deja lista E1 (se entrega antes de la Sesión 4).
```

## Cómo usar este notebook
- 🔍 **Prediga** antes de ejecutar · ✏️ **Complete** los `...` (ejecute celda por celda) · 💥 **Trampa**: hoy hay una muy buena · 🖼️ **Fichas** para guardar · ⚡ **Mini-reto** opcional.

> **Datos:** incidentes viales de Medellín, el periodo que decidimos en la Sesión 2: `df_grav` = enero 2019 – septiembre 2022 (antes del cambio de registro).


## 0. Setup

Igual que en la Sesión 1: baja al disco de Colab el CSV y `figuras.py` (las funciones que dibujan las figuras del curso). Si ya están, no los vuelve a bajar.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)

REPO = "https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/"
ARCHIVO = "incidentes_medellin_2019_2025.csv"
URL_REPO = REPO + "datos/" + ARCHIVO

In [ ]:
def asegurar_datos(archivo, url_repo):
    """Deja un archivo en el disco de Colab: lo baja del repositorio del curso si aún no está."""
    from pathlib import Path
    import requests
    if Path(archivo).exists():
        return True
    try:
        r = requests.get(url_repo, timeout=120)
        if r.ok:
            Path(archivo).write_bytes(r.content)
            return True
    except Exception:
        pass
    print(f"⚠️ No pude bajar {archivo} del repositorio. Use el Plan B (celda siguiente).")
    return False

asegurar_datos(ARCHIVO, URL_REPO)
asegurar_datos("figuras.py", REPO + "tools/figuras.py")
URL = ARCHIVO
try:
    import figuras as fg
except ModuleNotFoundError:   # Plan B: GitHub no respondió y no llegó figuras.py
    from google.colab import files
    print("⚠️ No llegó figuras.py: súbalo (el profesor lo comparte por el canal del curso).")
    files.upload()
    import figuras as fg
print("Listo. pandas", pd.__version__, "| datos en:", URL)

<details><summary><b>Plan B</b> — si la celda anterior avisó que no pudo bajar el archivo</summary>

La celda siguiente revisa sola si los datos llegaron; si no, le pedirá subir `incidentes_medellin_2019_2025.csv` desde su computador (el instructor lo comparte por el grupo del curso).
</details>

In [ ]:
# Plan B: solo actúa si la descarga falló. Con "Ejecutar todo" no se detiene.
from pathlib import Path
if Path(URL).exists():
    print("✓ Datos disponibles: no hace falta el Plan B.")
else:
    from google.colab import files
    print("⚠️ No se encontró", URL, "→ suba el archivo que compartió el instructor.")
    subido = files.upload()
    if subido:
        URL = next(iter(subido))
        print("Usando archivo local:", URL)

In [ ]:
fg.pipeline(["modelo", "evaluación"], modulo=1, subtitulo="HOY: entrenar, probar… y no hacerse trampa");

## 1. Las sesiones 1 y 2 en una celda

Cargar, limpiar nombres, construir la *y*, quedarse con el periodo correcto (`df_grav`), recodificar el `N/D` y armar X con *one-hot*. Todo lo que ya hizo, junto.

In [ ]:
import unicodedata

def limpiar_nombre(nombre):
    nombre = unicodedata.normalize("NFKD", nombre.strip().lower()).encode("ascii", "ignore").decode()
    return nombre.replace(" ", "_")

df = pd.read_excel(URL) if URL.endswith(".xlsx") else pd.read_csv(URL)
df.columns = [limpiar_nombre(c) for c in df.columns]
df = df.rename(columns={
    "llave": "id", "ano": "anio", "fecha_incidente": "fecha", "hora_incidente": "hora",
    "clase_incidente": "clase", "gravedad_incidente": "gravedad", "direccion_interpolada": "direccion",
    "codigo_comuna_planeacion": "codigo_comuna", "comuna_planeacion": "comuna",
    "codigo_barrio_planeacion": "codigo_barrio", "barrio_planeacion": "barrio",
})
df["fecha"] = pd.to_datetime(df["fecha"])
df["con_victimas"] = (df["gravedad"] != "SOLO DAÑOS").astype(int)

# Sesión 2: periodo donde se registraban los dos tipos de incidente
df_grav = df[(df["fecha"] >= "2019-01-01") & (df["fecha"] < "2022-10-01")].copy()
df_grav["hora_num"] = df_grav["hora"].astype(str).str.slice(0, 2).astype(int)
df_grav["dia_semana"] = df_grav["fecha"].dt.dayofweek
df_grav["mes"] = df_grav["fecha"].dt.month
df_grav = df_grav.replace({"comuna": {"N/D": np.nan}})

columnas_x = ["hora_num", "dia_semana", "mes", "comuna", "clase"]
X = pd.get_dummies(df_grav[columnas_x], columns=["comuna", "clase"], dtype=int, dummy_na=True)
y = df_grav["con_victimas"]
print("df_grav:", df_grav.shape, "| X:", X.shape, f"| {y.mean():.1%} con víctimas")

## 2. Separar: el examen que el modelo no puede ver

![¿Por qué separar entrenamiento y prueba?](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S03/img/examen_y_fuga.png)

En la Sesión 1 separamos sin explicar mucho. Hoy, despacio.

> 🧪 **Laboratorio** (para jugar después de clase, o ya si va adelantado): [**Entrenar y probar sin trampa**](https://nitobest.github.io/curso-aad-python/laboratorio/split.html) · separar antes de entrenar, la lotería de la partición y una variable "mágica" que acierta demasiado. Tres pestañas: **Mira**, **Juega** y **Reto** (prediga antes de correr). Funciona en el celular.

In [ ]:
fg.split_esquema();

### 👀 Mire
Vamos a separar con `test_size=0.2` y `stratify=y`. `stratify=y` obliga a que la proporción de la *y* sea igual en las dos partes: cuando corra la celda, compruebe cuántas filas quedan en prueba y qué % con víctimas tiene cada parte.

### ✏️ Completa
Complete los argumentos: 20 % para prueba, estratificado por `y`, y semilla 42 (para que a todos nos dé lo mismo).

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=..., stratify=..., random_state=42)   # TODO
print(f"entrenamiento: {len(X_train):>7,} filas   {y_train.mean():.1%} con víctimas".replace(",", " ").replace(".", ",").replace("%", " %"))
print(f"prueba:        {len(X_test):>7,} filas   {y_test.mean():.1%} con víctimas".replace(",", " ").replace(".", ",").replace("%", " %"))

## 3. Entrenar y mirar una predicción por dentro

El mismo árbol de profundidad 4 de las sesiones anteriores, y su línea base.

### 👀 Mire
¿Qué accuracy será **mayor**: el de entrenamiento (datos que el árbol ya vio) o el de prueba? Fíjese en la diferencia.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier

arbol = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)
base = DummyClassifier(strategy="most_frequent").fit(X_train, y_train).score(X_test, y_test)
print(f"línea base (prueba):     {base:.3f}")
print(f"árbol — entrenamiento:   {arbol.score(X_train, y_train):.3f}")
print(f"árbol — prueba:          {arbol.score(X_test, y_test):.3f}")

### Una fila entra, una predicción sale

El árbol entrenado no es una tabla de resultados: es una **función**. Le damos una fila de prueba (un choque real que nunca vio) y la seguimos por dentro.

In [ ]:
fila = X_test[X_test["clase_Choque"] == 1].iloc[0]
fg.camino_de_una_fila(arbol, fila, X.columns);

In [ ]:
print("Predicción del árbol:", "con víctimas" if arbol.predict(fila.to_frame().T)[0] == 1 else "solo daños")
print("Lo que pasó de verdad:", "con víctimas" if y_test.loc[fila.name] == 1 else "solo daños")

### ¿Qué hizo el `.fit()` para encontrar esas preguntas?

En la Sesión 1 vimos la curva de Gini y su valle. Eso **era optimización**, aunque no la llamamos así. Esta es la ficha que va a acompañar a cada algoritmo del curso:

![Dentro del .fit() — árbol de decisión](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S03/img/dentro_del_fit_arbol.png)

| Renglón | Árbol de decisión, con nuestros datos |
|---|---|
| **Qué minimiza** | la impureza de Gini de cada corte: $G = 1 - \sum_k p_k^2$ (Sesión 1) |
| **Moviendo qué** | la variable y el umbral de cada pregunta (`hora_num ≤ 7.5`, `clase_Choque ≤ 0.5`…) |
| **Cómo lo busca** | **búsqueda exhaustiva y voraz**: prueba todos los umbrales de todas las columnas, se queda con el mejor, repite en cada mitad y **nunca vuelve atrás** |
| **Qué perilla controla la búsqueda** | ninguna: no hay tasa de aprendizaje. `max_depth` y `min_samples_leaf` son perillas del **modelo** (cuánto puede crecer), no del buscador |

En la Sesión 4 aparece el contraste: la regresión lineal sí tiene un buscador con perilla (descenso de gradiente y su tasa de aprendizaje).

## 4. ¿Y si lo dejamos crecer sin límite? Sobreajuste

### 🖼️ Primero, con datos pequeños
Antes de hacerlo con 150 000 incidentes, mírelo con 260 puntos inventados de dos variables. Los puntos **llenos** son el entrenamiento; los **huecos**, la prueba (el árbol nunca los vio). El fondo es lo que predice cada árbol.

### 🔍 Predice
¿Cuál de los tres árboles acertará **más puntos huecos**: el de una pregunta, el de tres, o el que crece sin límite?

**Su predicción:** ____

In [ ]:
fg.memorizar_vs_aprender();

### Ahora con los datos reales

### 🔍 Predice
Entrenamos el mismo árbol **sin límite de profundidad** (`max_depth=None`): crece hasta que cada hoja sea pura o no se pueda partir más.
1. ¿El accuracy de **entrenamiento** sube o baja respecto a 0,766?
2. ¿Y el de **prueba** respecto a 0,769?
3. ¿Cuántas hojas cree que tendrá? (el de profundidad 4 tiene 14)

**Su predicción:** entrenamiento ____ · prueba ____ · hojas ≈ ____

In [ ]:
arbol_libre = DecisionTreeClassifier(max_depth=None, random_state=42).fit(X_train, y_train)
print(f"hojas: {arbol_libre.get_n_leaves():,}   profundidad: {arbol_libre.get_depth()}".replace(",", " "))
print(f"entrenamiento: {arbol_libre.score(X_train, y_train):.3f}   prueba: {arbol_libre.score(X_test, y_test):.3f}")

Ahora lo vemos en toda la escala: arriba, cómo se ven las regiones del árbol de dos variables (hora × día) a cada profundidad; abajo, el accuracy de entrenamiento y prueba del árbol completo.

In [ ]:
fg.arbol_profundidades(X_train, y_train, X_test, y_test, profundidades=(1, 2, 4, 8, 16, None));

### Lección del sobreajuste
- La curva azul (entrenamiento) **siempre sube**: más complejidad, más memoria.
- La curva roja (prueba) sube, se aplana y **cae**. El mejor modelo está donde la roja es máxima, no donde la azul.
- La **brecha** entre las dos curvas es el síntoma: si entrenamiento ≫ prueba, el modelo está memorizando.
- `max_depth` es la perilla que controla esa complejidad. En el Módulo 2 aparecen otras (`alpha`, `k`, número de árboles) con la misma lógica.

La forma general, que va a volver a ver en cada módulo (es un esquema, no datos):

In [ ]:
fg.brecha_train_test();

## 5. 💥 La variable que lo mejora todo

Una idea razonable: hay **calles más peligrosas** que otras. Si para cada incidente conocemos la *tasa de víctimas de su dirección* (qué proporción de los incidentes en esa dirección tuvo víctimas), el modelo debería predecir mejor. Esta técnica tiene nombre: ***target encoding*** (codificar una categoría con el promedio de la *y* en esa categoría). Se usa mucho cuando una categoría tiene demasiados valores para *one-hot* — `direccion` es una categórica de **altísima cardinalidad** (casi un identificador): en `df_grav` tiene más de 30 000 valores distintos.

Así funciona sobre una tabla pequeña:

In [ ]:
fg.target_encoding_paso_a_paso();

### 📐 Fundamento
Para cada dirección $c$ con $n_c$ incidentes, la variable nueva es el promedio de la *y* en esa dirección:

$$\text{tasa}_c = \frac{1}{n_c} \sum_{i \in c} y_i$$

Lea el símbolo $n_c$ con cuidado: es *cuántas veces aparece esa dirección*. Vuelve en un momento.

### 🔍 Predice
Si agregamos `tasa_dir` a X, ¿cuánto cree que sube el accuracy de prueba (hoy 0,769)?

**Su predicción:** ____

In [ ]:
# Tasa de víctimas de cada dirección, calculada con df_grav
tasa_por_direccion = df_grav.groupby("direccion")["con_victimas"].mean()

X_dir = X.copy()
X_dir["tasa_dir"] = df_grav["direccion"].map(tasa_por_direccion).fillna(y.mean())

arbol_dir = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_dir.loc[X_train.index], y_train)
print(f"entrenamiento: {arbol_dir.score(X_dir.loc[X_train.index], y_train):.3f}".replace(".", ","))
print(f"prueba:        {arbol_dir.score(X_dir.loc[X_test.index], y_test):.3f}".replace(".", ",") + "   (antes: 0,769)")

**0,816**: casi 5 puntos más, con un árbol igual de pequeño, y el entrenamiento y la prueba **iguales**, así que no hay sobreajuste. Parece el hallazgo de la sesión: *las calles peligrosas existen y el modelo las usa*.

Antes de celebrar, la prueba de fuego: **¿cuántas veces aparece cada dirección?**

In [ ]:
veces = df_grav["direccion"].value_counts()
n_una = (veces == 1).sum()
print(f"direcciones distintas: {len(veces):,}".replace(",", " "))
print(f"direcciones que aparecen UNA sola vez: {n_una:,}".replace(",", " ") + f" ({n_una / len(veces):.0%})".replace("%", " %"))

### 👀 Mire
Piense en una dirección que aparece **una sola vez** en los datos. ¿Cuánto vale su `tasa_dir`? ¿De qué depende? Con esa respuesta en la cabeza, mire la figura.

In [ ]:
fg.fuga_target_encoding(df_grav, "direccion");

In [ ]:
fg.target_encoding_paso_a_paso(destacar_unicas=True);

### Un paso en la dirección correcta: calcular solo con entrenamiento

Toda estadística que use la *y* se calcula **solo con las filas de entrenamiento**, y luego se *aplica* a las de prueba. Las direcciones de prueba que no aparecen en entrenamiento reciben la tasa general (no sabemos nada de ellas).

### ✏️ Completa

In [ ]:
filas_train = df_grav.loc[X_train.index]
tasa_solo_train = ...   # TODO: igual que tasa_por_direccion, pero con filas_train en vez de df_grav

X_dir_ok = X.copy()
X_dir_ok["tasa_dir"] = df_grav["direccion"].map(tasa_solo_train).fillna(y_train.mean())

arbol_ok = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_dir_ok.loc[X_train.index], y_train)
print("Árbol entrenado con tasa_dir calculada solo con entrenamiento (el resultado, en la figura de abajo).")

### 🔍 Predice
Calculada solo con entrenamiento, ¿la variable sube, deja igual o **baja** el accuracy de prueba respecto a 0,769 (el árbol sin la variable)?

**Su predicción:** ____ *(ejecute la celda de abajo después de predecir)*

In [ ]:
fg.fuga_resultados({
    "sin la variable": (arbol.score(X_train, y_train), arbol.score(X_test, y_test)),
    "tasa por dirección\ncon TODOS los datos": (arbol_dir.score(X_dir.loc[X_train.index], y_train), arbol_dir.score(X_dir.loc[X_test.index], y_test)),
    "tasa por dirección\nsolo con train": (arbol_ok.score(X_dir_ok.loc[X_train.index], y_train), arbol_ok.score(X_dir_ok.loc[X_test.index], y_test)),
}, linea_base=base);

<details><summary>¿Y la brecha 0,821 / 0,744?</summary>

Calcular la tasa solo con entrenamiento evita que el **examen** lleve su respuesta escrita, pero deja otra fuga, ahora **dentro** del entrenamiento: cada fila de entrenamiento usa una tasa calculada con *su propia y*. En las direcciones que aparecen una vez en entrenamiento, la tasa sigue siendo la respuesta copiada; por eso el entrenamiento sube a 0,821 y el árbol aprende a confiar en una columna que en prueba no sirve (0,744).

La versión profesional calcula la tasa de cada fila de entrenamiento **sin esa fila** (*cross-fitting*: parte el entrenamiento en bloques y a cada bloque le pone la tasa calculada con los otros). scikit-learn lo trae hecho:

```python
from sklearn.preprocessing import TargetEncoder
te = TargetEncoder(target_type="binary", random_state=42)
tasa_train = te.fit_transform(df_grav.loc[X_train.index, ["direccion"]], y_train)   # cada fila, sin su propia y
tasa_test = te.transform(df_grav.loc[X_test.index, ["direccion"]])                   # prueba: tasas de todo el entrenamiento
```

Con esa variable, el mismo árbol saca **0,767 en entrenamiento y 0,770 en prueba**: prácticamente lo mismo que sin la variable (0,766 / 0,769). Conclusión: la dirección no aporta nada, y **el 0,816 era fuga pura**.
</details>

### Lección de la trampa

![Cinco formas de hacerse trampa sin darse cuenta](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S03/img/fugas_comunes.png)

1. **Toda estadística que use la *y* se calcula solo con entrenamiento** (y, si se usa como variable, sin la fila misma: `TargetEncoder`). Tasas, medias por grupo, conteos de positivos.
2. Si una variable mejora la prueba "demasiado", aplique la prueba de fuego: *¿la tendría antes de conocer la y?*
3. Que entrenamiento y prueba den igual **no** prueba que no haya fuga: con fuga, los dos se inflan juntos.

## 6. La pregunta de la Sesión 2: ¿el escalador y la mediana también hacían trampa?

En la Sesión 2 calculamos la mediana de las coordenadas y la escala de `hora_num` con **todos** los datos. Es la forma 3 de la ficha ("preparar antes de separar"). La regla es la misma que con la tasa: **`.fit()` solo con entrenamiento, `.transform()` en las dos partes.**

In [ ]:
fg.fit_solo_con_train();

In [ ]:
from sklearn.preprocessing import StandardScaler

escalador = StandardScaler().fit(X_train[["hora_num"]])          # aprende media y desviación SOLO de train
hora_train_z = escalador.transform(X_train[["hora_num"]])
hora_test_z = escalador.transform(X_test[["hora_num"]])           # aplica lo aprendido a prueba
print(f"media aprendida con train: {escalador.mean_[0]:.3f}  |  media de todo X: {X['hora_num'].mean():.3f}")

La diferencia es mínima (la media de la hora casi no cambia por quitar el 20 %), y a un árbol el escalado ni siquiera le importa. **¿Entonces por qué tanto cuidado?** Porque con la tasa por dirección vimos que la misma operación —"calcular algo con todos los datos"— puede ir de inofensiva a devastadora según *qué* se calcula. El hábito correcto es uno solo y no depende del caso: separar primero, aprender después. En el Módulo 4 el `Pipeline` de scikit-learn lo hace automático.

## 7. Lo que el accuracy no dice

El árbol acierta el 77 %. ¿Pero en **qué** se equivoca? No es lo mismo decir "solo daños" a un incidente con heridos que lo contrario.

In [ ]:
pred = arbol.predict(X_test)
pd.crosstab(y_test.map({0: "solo daños", 1: "con víctimas"}),
            pd.Series(pred, index=y_test.index).map({0: "solo daños", 1: "con víctimas"}),
            rownames=["realidad"], colnames=["predicción del árbol"])

Esta tabla tiene nombre —**matriz de confusión**— y en la Sesión 5 la vamos a necesitar en serio, cuando aparezca un dataset donde el accuracy engaña por completo. Por ahora solo mírela: ¿qué error comete más el árbol?

## 8. 📦 Entrega E1 — su dataset, mirado con lupa

Se entrega **antes de la Sesión 4, por el canal que indique el profesor**. Es un notebook en Colab con su dataset y estas secciones, cada una de pocas líneas:

| # | Sección | Qué debe contener |
|---|---|---|
| 1 | Dataset | Nombre, fuente (enlace), licencia si la tiene, y **qué es una fila** en una frase |
| 2 | Pregunta de ML | Qué quiere predecir o descubrir, ¿es clasificación, regresión o no supervisado?, y por qué le importaría a alguien |
| 3 | La *y* | Regla exacta que la define, su proporción (clases) o su rango (número) |
| 4 | `explorar()` | La tabla de la Sesión 2 sobre su dataset, y un comentario de cada `alerta` |
| 5 | En el tiempo | Si tiene fechas: la *y* graficada por mes. ¿Hay saltos? ¿De realidad o de registro? |
| 6 | **Columnas sospechosas** | Lista de columnas que se conocen **después** de la *y* o que podrían copiarla (ficha de fugas). Una línea de justificación por columna |
| 7 | Decisiones | Filas, periodos o columnas que descartó y por qué |

Rúbrica (1–4 cada criterio): corrección técnica · ausencia de fuga / split correcto · interpretación honesta · claridad del notebook. Detalle en `proyecto/README.md` del repositorio.

La función `explorar()` de la Sesión 2, para usarla aquí con su dataset:

In [ ]:
SOSPECHOSOS = ["N/D", "ND", "SIN INFORMACION", "NO APLICA", "NO REPORTA", "0", "-", ""]

def explorar(tabla, y=None, sospechosos=SOSPECHOSOS):
    """Resumen de una tabla antes de modelar: una fila por columna (Sesión 2)."""
    print(f"Filas: {len(tabla):,}   Columnas: {tabla.shape[1]}".replace(",", " "))
    r = pd.DataFrame(index=tabla.columns)
    r["dtype"] = tabla.dtypes.astype(str)
    r["n_distintos"] = tabla.nunique()
    r["%_vacios"] = (tabla.isna().mean() * 100).round(1)
    texto = tabla.select_dtypes(include=["object", "string"])
    disfr = {c: texto[c].astype(str).str.strip().str.upper().isin(sospechosos).mean() * 100 for c in texto}
    r["%_disfrazados"] = pd.Series(disfr).reindex(tabla.columns).fillna(0).round(1)
    r["ejemplo"] = tabla.iloc[0].astype(str).str.slice(0, 25)
    r["alerta"] = ""
    r.loc[r["n_distintos"] == len(tabla), "alerta"] = "¿identificador?"
    r.loc[r["dtype"].isin(["object", "str", "string"]) & (r["n_distintos"] > 0.1 * len(tabla)), "alerta"] = "¿alta cardinalidad? ¿memorizable?"
    r.loc[r["%_vacios"] + r["%_disfrazados"] > 5, "alerta"] += " ¿por qué falta?"
    if y is not None:
        print(f"\nVariable objetivo «{y}»:")
        print(tabla[y].value_counts(normalize=True).round(3) if tabla[y].nunique() <= 10 else tabla[y].describe().round(2))
    return r

# Ejemplo con los datos de clase. Para E1: cargue su dataset (README → "cargar sus datos desde Drive") y cambie esta línea.
explorar(df_grav[["fecha", "hora", "clase", "direccion", "latitud", "comuna", "barrio", "con_victimas"]], y="con_victimas")

Fíjese que `direccion` ahora sale con la alerta *¿alta cardinalidad? ¿memorizable?*. Después de hoy sabe exactamente por qué.

## ⚡ Mini-reto (opcional)

Repita la trampa con `barrio` en vez de `direccion`: tasa de víctimas por barrio, calculada con todos los datos y luego solo con entrenamiento.

1. ¿La fuga se nota? ¿Por qué cree que no (o sí)?
2. Pista: ¿cuántas filas tiene en promedio cada barrio? ¿Y cada dirección?

<details><summary>Código de partida</summary>

```python
for nombre, tasa in [("todos los datos", df_grav.groupby("barrio")["con_victimas"].mean()),
                     ("solo train", filas_train.groupby("barrio")["con_victimas"].mean())]:
    Z = X.copy(); Z["tasa_barrio"] = df_grav["barrio"].map(tasa).fillna(y_train.mean())
    m = DecisionTreeClassifier(max_depth=4, random_state=42).fit(Z.loc[X_train.index], y_train)
    print(nombre, round(m.score(Z.loc[X_test.index], y_test), 3))
```
</details>

In [ ]:
# Su código aquí

## 9. Cierre del Módulo 1

| Etapa | S1 | S2 | S3 (hoy) |
|---|---|---|---|
| Datos | cargar y mirar | tipos, vacíos, la *y* en el tiempo, `df_grav` | — |
| Preprocesamiento | dummies, `dropna` | imputar + marcar, one-hot, escalar | **`.fit()` solo con train**; *target encoding* sin fuga |
| Modelo | árbol | (detector de problemas) | **sobreajuste**: profundidad vs memoria · Dentro del `.fit()` |
| Evaluación | accuracy vs línea base | % del error del tonto que se elimina | **train vs test**, `stratify`, **fuga de información** |
| Interpretación | reglas e importancias | ¿cambió el mundo o el dato? | **una fila por dentro** del árbol |

**Tres frases para llevarse del Módulo 1:**
1. *El problema casi nunca es el algoritmo* (S1–S2).
2. *Menos accuracy, modelo más honesto*: un número solo no dice nada; se compara con la línea base y sobre los datos correctos (S2).
3. *Separar primero, aprender después*: todo lo que use la *y* se calcula con entrenamiento (S3).

**Para la Sesión 4 — Módulo 2, regresión:** cambiamos de dataset y de pregunta: ya no "¿tendrá víctimas?" sino "¿**cuánto**?". Entregue E1 antes de la sesión.